# Notebook 06 - Evaluate Language Adherence

Question: did the model actually answer in Kirundi/Rundi?

The preferred automatic evaluator is AfroLID (`UBC-NLP/afrolid_1.5`), an African-language-aware language identification model. If that model is too heavy for your machine, this notebook can use a transparent heuristic fallback. The fallback is not a real language ID model.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.append(str(ROOT / "src"))

from kirundi_sft_starter.evals import classify_language, load_response_table, summarize_language_adherence
from kirundi_sft_starter.utils import ensure_dir, load_yaml

config = load_yaml(ROOT / "configs/project.yaml")
USE_AFROLID = False


## Generate responses for the language prompts

```bash
python scripts/generate_model_responses.py --model-key base
python scripts/generate_model_responses.py --model-key sft_raw
python scripts/generate_model_responses.py --model-key sft_adapted
```

Use `--dry-run` to create placeholder files without Tinker credentials.

In [ ]:
frames = []
for model_key, model_cfg in config["models"]["registry"].items():
    path = ROOT / model_cfg["response_path"]
    if path.exists():
        frames.append(load_response_table(path, model_key))

if not frames:
    raise FileNotFoundError("No response files found. Generate responses first.")

responses = pd.concat(frames, ignore_index=True)
responses.head()

In [ ]:
lid = None
if USE_AFROLID:
    from transformers import pipeline
    lid = pipeline("text-classification", model=config["evaluation"]["language_id_model"])

detected = responses["response"].apply(lambda text: classify_language(text, lid)).apply(pd.Series)
scored = pd.concat([responses, detected], axis=1)
summary = summarize_language_adherence(scored)
display(summary)

output_path = ROOT / config["evaluation"]["language_results_path"]
ensure_dir(output_path)
summary.to_csv(output_path, index=False)
print("Saved", output_path)

## Report table

| model | num_prompts | % Kirundi/Rundi responses | notes |
|---|---:|---:|---|
| base | filled by notebook | filled by notebook | automatic LID or fallback |
| sft_raw | filled by notebook | filled by notebook | automatic LID or fallback |
| sft_adapted | filled by notebook | filled by notebook | automatic LID or fallback |